In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/cleaned.csv")
print(df.shape)

# Log-transform the target (price is skewed — mention this in the report)
y = np.log1p(df["price"])
X = df.drop(columns=["price"])

# Split: categorical columns stay as strings, we'll encode them in a pipeline
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)


(370554, 15)
(296443, 14) (74111, 14)


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np

cat_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object", "str"]).columns.tolist()

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            max_categories=15,
            sparse_output=False,      # ← boolean, no quotes
            dtype=np.float32,         # ← halves memory vs float64
        )),
    ]), cat_cols),
])

In [3]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(model, name):
    model.fit(X_train, y_train)
    pred_log = model.predict(X_test)
    pred = np.expm1(pred_log)          # back to dollars
    true = np.expm1(y_test)
    mae = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    r2 = r2_score(true, pred)
    print(f"{name:20} MAE=${mae:,.0f}  RMSE=${rmse:,.0f}  R²={r2:.3f}")
    return model

baseline = evaluate(DummyRegressor(strategy="median"), "Baseline (median)")

Baseline (median)    MAE=$11,197  RMSE=$14,895  R²=-0.055


In [4]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import joblib

models = {
    "Ridge": Ridge(),
    "RandomForest": RandomForestRegressor(
        n_estimators=100,
        max_depth=15,          # the fix: bounded trees
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42,
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=300,
        early_stopping=True,
        random_state=42,
    ),
}

results = {}
for name, m in models.items():
    print(f"--- training {name} ...", flush=True)
    pipe = Pipeline([("prep", preprocess), ("model", m)])
    results[name] = evaluate(pipe, name)
    joblib.dump(results[name], f"../app/model_{name}.pkl")
    print(f"    saved ../app/model_{name}.pkl", flush=True)


--- training Ridge ...
Ridge                MAE=$4,928  RMSE=$8,202  R²=0.680
    saved ../app/model_Ridge.pkl
--- training RandomForest ...
RandomForest         MAE=$3,699  RMSE=$6,593  R²=0.793
    saved ../app/model_RandomForest.pkl
--- training HistGradientBoosting ...
HistGradientBoosting MAE=$3,855  RMSE=$6,692  R²=0.787
    saved ../app/model_HistGradientBoosting.pkl


In [5]:
# compare on MAE (lower is better)
for name, mdl in results.items():
    print(name, "saved")

best_name = min(results, key=lambda n: __import__("sklearn.metrics", fromlist=["mean_absolute_error"]).mean_absolute_error(
    np.expm1(y_test), np.expm1(results[n].predict(X_test))
))
print("BEST:", best_name)
joblib.dump(results[best_name], "../app/model.pkl", compress = 3)

Ridge saved
RandomForest saved
HistGradientBoosting saved
BEST: RandomForest


['../app/model.pkl']

In [6]:
import joblib
joblib.dump(preprocess, "../app/preprocess.pkl")
print("saved app/preprocess.pkl")


saved app/preprocess.pkl


In [7]:
import json
options = {c: sorted(X[c].dropna().unique().tolist())
           for c in X.select_dtypes(include=["object", "str"]).columns}
with open("../app/options.json", "w") as f:
    json.dump(options, f)
print({k: len(v) for k, v in options.items()})
print(X[["year", "odometer", "cylinders", "posting_month"]].describe())


{'manufacturer': 41, 'model': 31, 'condition': 6, 'fuel': 5, 'title_status': 6, 'transmission': 3, 'drive': 3, 'type': 13, 'paint_color': 12, 'state': 51}
                year       odometer      cylinders  posting_month
count  370554.000000  370554.000000  368670.000000  370554.000000
mean     2012.289607   93594.702683       5.892912       4.253728
std         5.816610   63305.372443       1.522487       0.435144
min      1990.000000       1.000000       3.000000       4.000000
25%      2009.000000   39192.250000       4.000000       4.000000
50%      2013.000000   88817.000000       6.000000       4.000000
75%      2017.000000  136400.000000       8.000000       5.000000
max      2022.000000  400000.000000      12.000000       5.000000
